# 📖 Notebook 2: Load Balancing Algorithms Compared

In Notebook 1 we used plain **round robin**. It's simple and fair in terms of
*request count*, but it doesn't adapt when some requests are much more
expensive than others.

In this notebook we compare five classic algorithms on the same workload:

1. **Round Robin** — cycle through the list.
2. **Weighted Round Robin** — give stronger backends more turns.
3. **Least Connections** — send to whoever has the fewest in-flight requests.
4. **Random** — pick any backend uniformly at random.
5. **Power of Two Choices** — pick two at random, send to the less busy one.

## Learning Objectives

By the end of this notebook, you'll understand:

- How each algorithm decides where to send a request
- Why "fair by request count" ≠ "fair by load"
- What **tail latency** means and why it matters more than the average
- When to pick which algorithm in the real world


## 🛠️ Setup

From the lab folder:

```bash
cd 01-foundations/load-balancing
uv sync
```

### Kernel selection

In VS Code, click the kernel picker at the **top-right** of this notebook and
choose the `.venv` interpreter (it will be named something like
`.venv (Python 3.x)`).

If the `.venv` kernel doesn't appear in the list, reload the VS Code window:

- `Cmd+Shift+P` (macOS) or `Ctrl+Shift+P` (Windows/Linux)
- Type and select **"Developer: Reload Window"**

No Docker or external services are needed for this lab — everything runs
in-process with plain Python.


## 🧱 Setting the stage

We'll reuse the idea of fake backends from Notebook 1, but this time each
backend has a **capacity** (how many "units" of work per second it can do).
Stronger backends finish the same work in less wall-clock time.

We'll also track **active_connections** — the number of requests currently
in-flight — so the least-connections algorithm has something to look at.


In [ ]:
from pydantic import BaseModel, Field
from typing import Literal
import random
import itertools
import heapq

class Request(BaseModel):
    """An incoming request with an amount of 'work' measured in work-units."""
    id: int
    work_units: float = Field(gt=0)


class Backend(BaseModel):
    """A pretend backend.

    `capacity` = how many work-units it can process per second. A capacity of
    2.0 means it finishes a 1-unit request in 0.5 seconds.
    """
    name: str
    capacity: float = Field(gt=0)
    active: int = 0             # in-flight requests (queued + running)
    free_at: float = 0.0        # sim time at which the single worker is free again
    total_latency: float = 0.0  # sum of response times (for stats)
    handled: int = 0

    def service_time(self, req: Request) -> float:
        """How long this backend would take to process `req`, alone."""
        return req.work_units / self.capacity


# A heterogeneous fleet — 2 beefy servers, 2 small ones
def fresh_backends() -> list[Backend]:
    return [
        Backend(name="big-1",   capacity=4.0),
        Backend(name="big-2",   capacity=4.0),
        Backend(name="small-1", capacity=1.0),
        Backend(name="small-2", capacity=1.0),
    ]


fresh_backends()


## 🎲 Building a realistic, uneven workload

Real traffic is **bursty** and **skewed**: most requests are small, but a few
are huge (think: a search query vs. generating a PDF report). We'll build a
workload that looks like that.


In [ ]:
random.seed(7)

def make_workload(n: int = 2000) -> list[Request]:
    """90% small requests, 10% 'heavy' requests (5-20x bigger)."""
    reqs = []
    for i in range(n):
        if random.random() < 0.10:
            work = random.uniform(5.0, 20.0)   # heavy
        else:
            work = random.uniform(0.1, 1.0)    # light
        reqs.append(Request(id=i, work_units=work))
    return reqs


workload = make_workload()
heavy = sum(1 for r in workload if r.work_units >= 5)
print(f"{len(workload)} requests, {heavy} of them are heavy ({heavy/len(workload):.0%})")
print(f"Total work: {sum(r.work_units for r in workload):.1f} units")


## ⏱️ A tiny event-driven simulator

To compare algorithms we need to know **when** each request finishes, not
just "who got it". We'll simulate time with a priority queue of events:

- Every `ARRIVAL_INTERVAL` seconds a new request arrives.
- The load balancer picks a backend using its algorithm.
- **Each backend runs one request at a time.** If it's already busy, the new
  request **waits in that backend's queue**. This is the whole reason load
  balancing matters — without a queue, a "busy" backend costs nothing and every
  algorithm looks identical.
- When a request finishes, the backend's `active` counter drops.

Latency for a request = `finish_time - arrival_time` = **queueing time + service
time**. That's what users feel.

### Sizing the experiment

The fleet can do `4 + 4 + 1 + 1 = 10` work-units per second. Our workload is
~3,500 units, so we space arrivals `0.5 s` apart: 2000 × 0.5 s = 1000 s of
simulated time, i.e. the fleet is offered ~35% of what it can do overall.

Keep that number in mind — it means **no algorithm is forced to fail**. If the
tail latency still explodes, that is the algorithm's own doing, not overload.

In [ ]:
ARRIVAL_INTERVAL = 0.5   # seconds between request arrivals

def simulate(workload, pick_backend, interval: float = ARRIVAL_INTERVAL):
    """Run `workload` through a load balancer that calls `pick_backend(backends, req)`.

    Each backend is a single-threaded worker with a FIFO queue, so a request that
    arrives at a busy backend waits for the ones ahead of it.

    Returns:
      - the list of backends with stats filled in
      - a list of per-request latencies (seconds), in finish order
    """
    backends = fresh_backends()
    tiebreak = itertools.count()
    events: list[tuple[float, int, str, object]] = []

    for req in workload:
        heapq.heappush(events, (req.id * interval, next(tiebreak), "arrive", req))

    arrival_of: dict[int, float] = {req.id: req.id * interval for req in workload}
    latencies: list[float] = []  # per-request latency, in finish order

    while events:
        now, _, kind, payload = heapq.heappop(events)
        if kind == "arrive":
            req: Request = payload
            b = pick_backend(backends, req)
            b.active += 1
            b.handled += 1
            # The backend can only start once it has finished whatever it was doing.
            start = max(now, b.free_at)
            finish_time = start + b.service_time(req)
            b.free_at = finish_time
            heapq.heappush(events, (finish_time, next(tiebreak), "finish", (b, req)))
        else:  # finish
            b, req = payload
            b.active -= 1
            latency = now - arrival_of[req.id]
            b.total_latency += latency
            latencies.append(latency)

    return backends, latencies


def report(name, backends, latencies):
    for b in backends:
        avg = b.total_latency / b.handled if b.handled else 0
        print(f"{b.name:8s} handled={b.handled:5d}  avg_latency={avg:7.2f}s")
    print(f"{'':8s} → fleet p99 = {sorted(latencies)[int(0.99 * len(latencies)) - 1]:.2f}s")

## 1️⃣ Round Robin (baseline)

Same as Notebook 1: cycle through the list. Notice it has **no idea** that
`big-1` is 4× stronger than `small-1` — it treats them identically.


In [ ]:
class RoundRobinPicker:
    def __init__(self):
        self.i = 0
    def __call__(self, backends, req):
        b = backends[self.i % len(backends)]
        self.i += 1
        return b

rr_backends, rr_lat = simulate(workload, RoundRobinPicker())
report("round robin", rr_backends, rr_lat)

# Round robin is perfectly fair *by request count* — that is exactly the problem.
assert len({b.handled for b in rr_backends}) == 1, "round robin must hand out equal counts"

## 2️⃣ Weighted Round Robin

If `big-1` is 4× as strong as `small-1`, give it 4× as many turns. We encode
that by building a list where each backend appears `capacity` times, then
cycle through it.

This is what real load balancers like NGINX and HAProxy do when you set a
`weight=N` on a backend — though they use a *smooth* variant that interleaves the
slots (`big-1, big-2, big-1, small-1, ...`) instead of emitting them in runs. The
distribution is the same; the smooth version just avoids sending four requests to
the same backend back-to-back.

The weights below are `4, 4, 1, 1`, so the split must come out **40 / 40 / 10 / 10**.
We assert that rather than eyeballing it.

In [ ]:
class WeightedRoundRobinPicker:
    def __init__(self, backends_template):
        # Expand the list: big-1 appears 4x, small-1 once, etc.
        self.slots = []
        for b in backends_template:
            self.slots.extend([b.name] * int(b.capacity))
        self.i = 0

    def __call__(self, backends, req):
        name = self.slots[self.i % len(self.slots)]
        self.i += 1
        # Look up the live backend object by name
        return next(b for b in backends if b.name == name)


wrr_backends, wrr_lat = simulate(workload, WeightedRoundRobinPicker(fresh_backends()))
report("weighted round robin", wrr_backends, wrr_lat)

# The weights must actually produce the claimed distribution.
total = sum(b.handled for b in wrr_backends)
shares = {b.name: b.handled / total for b in wrr_backends}
expected = {"big-1": 0.4, "big-2": 0.4, "small-1": 0.1, "small-2": 0.1}
print("\nshare of requests:", {k: f"{v:.1%}" for k, v in shares.items()})
for name, want in expected.items():
    assert abs(shares[name] - want) < 0.005, (name, shares[name], want)

## 3️⃣ Least Connections

> Send the next request to whichever backend currently has the **fewest
> in-flight requests**.

This adapts automatically without anyone configuring weights: if `small-1` gets
stuck behind a huge request, its `active` counter stays high and the LB routes
around it.

Two details that matter:

- **Ties are broken at random.** With a deterministic tie-break (say, "first in the
  list wins") an idle fleet sends *everything* to `big-1`, which looks like a
  spectacular result but is really just an artefact.
- **Plain least-connections counts connections, not capacity.** It has no idea
  `big-1` is 4× stronger, so on a *heterogeneous* fleet it converges to roughly
  equal request counts — which is round robin's problem all over again. We'll fix
  that in a moment with the weighted variant.

In [ ]:
def least_connections(backends, req):
    """Fewest in-flight requests wins; ties broken uniformly at random."""
    fewest = min(b.active for b in backends)
    return random.choice([b for b in backends if b.active == fewest])

random.seed(99)
lc_backends, lc_lat = simulate(workload, least_connections)
report("least connections", lc_backends, lc_lat)

# It must actually differentiate the backends — if every count is identical it has
# silently degenerated into round robin and the demo is proving nothing.
counts = [b.handled for b in lc_backends]
assert len(set(counts)) > 1, "least-connections degenerated to round robin"
print(f"\nconnection-count spread: {min(counts)}..{max(counts)} "
      f"(round robin would be {len(workload)//len(lc_backends)} for everyone)")

### 3️⃣b Weighted Least Connections

The fix is one character of arithmetic: score each backend by
**`active / capacity`** instead of `active`. Now "two requests queued on a 4×
server" ranks as less loaded than "one request queued on a 1× server", which is
true.

This is what Envoy calls `LEAST_REQUEST` with weights, and NGINX Plus calls
`least_conn` with `weight=`. It needs the same in-flight counter as plain
least-connections, plus a static weight per backend.

Fair warning before you run it: on *this* workload it will barely help. Keep
reading — the reason is more interesting than the result.

In [ ]:
def weighted_least_connections(backends, req):
    """Least *normalised* load: in-flight requests per unit of capacity."""
    lowest = min(b.active / b.capacity for b in backends)
    return random.choice([b for b in backends if b.active / b.capacity == lowest])

random.seed(99)
wlc_backends, wlc_lat = simulate(workload, weighted_least_connections)
report("weighted least connections", wlc_backends, wlc_lat)

## 4️⃣ Random

Pick any backend uniformly at random. Sounds dumb, but it actually works
*surprisingly* well at scale because the law of large numbers smooths things
out. And it needs **zero shared state** — every LB node can decide
independently, which matters if you have a fleet of load balancers.


In [ ]:
def random_pick(backends, req):
    return random.choice(backends)

random.seed(123)
rand_backends, rand_lat = simulate(workload, random_pick)
report("random", rand_backends, rand_lat)

## 5️⃣ Power of Two Choices (P2C)

> Pick **two** backends at random, then send the request to whichever one is
> less busy.

This sounds almost as silly as plain random, but mathematically it's a huge
upgrade. The maximum load on any backend drops from `O(log N / log log N)`
(plain random) to `O(log log N)` — a famous result by Mitzenmacher (2001).

P2C is the algorithm of choice in modern load balancers (NGINX's
`random two least_conn`, Envoy's `LEAST_REQUEST` with `choice_count=2`, HAProxy's
`first` with `random`). Why? It needs **almost no shared state** (each LB just
picks two backends and reads their counter), but behaves nearly as well as full
least-connections.

Because our fleet has **uneven capacity**, we score by `active / capacity` so
P2C doesn't blindly favor small backends just because they have fewer
connections.


In [ ]:
def power_of_two(backends, req):
    a, b = random.sample(backends, 2)
    # Score by load-per-capacity so unequal backends are compared fairly.
    score = lambda x: x.active / x.capacity
    return a if score(a) <= score(b) else b

random.seed(321)
p2c_backends, p2c_lat = simulate(workload, power_of_two)
report("power of two choices", p2c_backends, p2c_lat)

## 🏁 Head-to-head comparison

Now that the simulator records the latency of **every individual request**, we
can compute the real metrics users care about:

- **avg** — the typical user experience.
- **p50** (median) — half of users are faster than this.
- **p95** — 95% of users are faster than this. This is the "tail". Tail
  latency is what makes a site feel slow even when the average looks fine,
  because each page view usually makes many requests and the slowest one
  dominates.
- **p99** — only 1% of users wait longer than this. Important for SLAs.


In [ ]:
import statistics

def percentiles(latencies: list[float]) -> dict[str, float]:
    """Return avg + key percentiles for a list of per-request latencies."""
    s = sorted(latencies)
    def pct(p):
        # Nearest-rank percentile — fine for a teaching notebook.
        k = max(0, min(len(s) - 1, int(round(p / 100.0 * len(s))) - 1))
        return s[k]
    return {
        "avg": statistics.mean(s),
        "p50": pct(50),
        "p95": pct(95),
        "p99": pct(99),
    }

results = {
    "round_robin":          percentiles(rr_lat),
    "weighted_round_robin": percentiles(wrr_lat),
    "least_connections":    percentiles(lc_lat),
    "weighted_least_conn":  percentiles(wlc_lat),
    "random":               percentiles(rand_lat),
    "power_of_two":         percentiles(p2c_lat),
}

print(f"{'algorithm':22s} {'avg':>8s} {'p50':>8s} {'p95':>8s} {'p99':>8s}")
print("-" * 58)
for name, m in results.items():
    print(f"{name:22s} {m['avg']:8.2f} {m['p50']:8.2f} {m['p95']:8.2f} {m['p99']:8.2f}")

# --- The claims this notebook makes, as assertions ---
# 1. Capacity-blind algorithms (round robin, random) drown the small backends even
#    though the *fleet* is only ~35% utilised.
assert results["round_robin"]["p99"] > 4 * results["weighted_round_robin"]["p99"]
assert results["random"]["p99"] > 4 * results["weighted_round_robin"]["p99"]

# 2. Reacting to live load beats a static weight on the typical request and at p95.
for algo in ("least_connections", "weighted_least_conn"):
    assert results[algo]["avg"] < results["weighted_round_robin"]["avg"]
    assert results[algo]["p95"] < results["weighted_round_robin"]["p95"]

# 3. P2C gets most of that benefit from two random samples and no global view.
assert results["power_of_two"]["avg"] < 0.2 * results["round_robin"]["avg"]

# 4. But every capacity-aware algorithm lands on essentially the SAME p99.
p99s = [results[a]["p99"] for a in
        ("weighted_round_robin", "least_connections", "weighted_least_conn", "power_of_two")]
assert max(p99s) / min(p99s) < 1.5, p99s
print(f"\ncapacity-aware p99s all land in {min(p99s):.1f}s..{max(p99s):.1f}s")

### 🤨 Why does every decent algorithm have the same p99?

Look at the last line of output. Weighted round robin, least-connections, weighted
least-connections and P2C all bottom out around the same p99 — even though their
averages differ by 50%.

The reason: **none of these algorithms look at the request**. They see "a request
arrived", not "a 20-work-unit monster arrived". So sooner or later a 20-unit request
lands on a 1-unit backend and takes 20 seconds no matter who routed it. That sets a
**floor** on the tail:

```
p99 floor ≈ largest request / smallest backend capacity = 20 / 1 = 20 s
```

Improving the tail past that floor needs a different lever entirely — smaller
backends removed from the pool, request-size hints, admission control, or splitting
heavy work onto a separate queue. Picking a cleverer load-balancing algorithm will
not do it. This is a genuinely useful thing to know before you spend a sprint
tuning your LB config.

### 🔥 When *does* weighting the connection count pay off?

At 35% fleet utilisation the backends are usually idle when a request shows up, so
`active` is 0 almost everywhere and `active / capacity` has nothing to distinguish.
The two variants only diverge once queues get deep.

Let's turn the pressure up to ~80% utilisation by shrinking the gap between arrivals,
and run just those two.

In [ ]:
BUSY_INTERVAL = 0.22   # ~80% fleet utilisation instead of ~35%

busy = {}
for label, picker in [("least_connections", least_connections),
                      ("weighted_least_conn", weighted_least_connections)]:
    random.seed(99)
    _, lat = simulate(workload, picker, interval=BUSY_INTERVAL)
    busy[label] = percentiles(lat)

print(f"{'algorithm':22s} {'avg':>8s} {'p95':>8s} {'p99':>8s}   (fleet ~80% utilised)")
print("-" * 60)
for name, mtr in busy.items():
    print(f"{name:22s} {mtr['avg']:8.2f} {mtr['p95']:8.2f} {mtr['p99']:8.2f}")

# Under pressure, normalising by capacity is worth ~30% at p95 and ~1.5x at p99.
assert busy["weighted_least_conn"]["p95"] < 0.85 * busy["least_connections"]["p95"]
assert busy["weighted_least_conn"]["p99"] < 0.85 * busy["least_connections"]["p99"]
print("\nWeighting the counter only matters once the queues are non-empty —"
      "\nwhich is exactly when you care.")

In [ ]:
import matplotlib.pyplot as plt

names = list(results.keys())
avgs  = [results[n]["avg"] for n in names]
p99s  = [results[n]["p99"] for n in names]

fig, ax = plt.subplots(figsize=(10, 4))
x = range(len(names))
ax.bar([i - 0.2 for i in x], avgs, width=0.4, label="average latency", color="#4C9AFF")
ax.bar([i + 0.2 for i in x], p99s, width=0.4, label="p99 latency",     color="#FF5630")
ax.set_xticks(list(x))
ax.set_xticklabels(names, rotation=20, ha="right")
ax.set_ylabel("seconds")
ax.set_yscale("log")   # log scale: round robin is off the chart otherwise
ax.set_title("Algorithm comparison (log scale, lower is better)")
ax.legend()
plt.tight_layout()
plt.show()


## 🧠 Takeaways & when to use what

| Algorithm | Best when… | Watch out for… |
|---|---|---|
| **Round Robin** | backends are identical *and* requests are uniform | a heterogeneous fleet — the weak nodes queue up while the strong ones idle |
| **Weighted Round Robin** | backends have known, different capacities | capacities change over time (autoscaling, noisy neighbours); still blind to *this* request being huge |
| **Least Connections** | requests have very different costs and backends are identical | on an uneven fleet it is barely better than round robin — it counts connections, not capacity |
| **Weighted Least Connections** | uneven fleet **and** uneven requests | the LB must track in-flight state per backend, so every LB node needs its own view |
| **Random** | very large fleets, or stateless LB nodes | small fleets → high variance; same capacity-blindness as round robin |
| **Power of Two Choices (P2C)** | you want least-connections quality with almost no coordination | needs at least 2 backends; weight by capacity if the fleet is uneven |

### The thing to actually remember

Look at the p99 column, then look back at the sizing note: the fleet was only
**~35% utilised**. Round robin still produced a p99 in the *tens of seconds*,
because it kept feeding 20-unit requests to 1-unit backends. Tail latency is
rarely about "not enough servers" — it is usually about **work landing on the
wrong server**.

> 💡 In real life, most modern L7 load balancers (NGINX, Envoy, HAProxy, AWS
> ALB) default to **least connections** or **power-of-two-choices**. P2C is
> the secret sauce behind a lot of "magic" load balancers — it's almost as good
> as least-connections but works without any coordination between LB instances.

👉 Next: in **Notebook 3** we'll see what happens when a backend gets sick,
how **health checks** remove it from the pool, and why **sticky sessions**
are a double-edged sword. Then **Notebook 4** tackles **consistent hashing**
— how to do sticky routing without losing everyone's session every time you
add or remove a backend.